# Chessathon NNUE training

Trains `(768 -> 256) x 2 -> 1` SCReLU with 8 output buckets, then quantises to int16 and writes
`net256.npz` -- the file the engine loads. Nothing here ships; only the `.npz` does.

**Before running:** set *Runtime -> Change runtime type -> T4 GPU*, and put the shard files in
Google Drive under `MyDrive/chessathon/shards/`. Records were scattered across the 64 shards
uniformly at random when they were written, so **any subset is already a uniform random sample** of
the whole database and needs no reshuffling -- which is what makes uploading a subset legitimate
rather than a compromise.

**Upload 32 of them: `shard00.bin` to `shard31.bin`, 6.05 GB, 189M positions.** The sizing is set
by the width A/B, not by the 256-wide net:

| shards | size | positions | epochs at 60k steps | positions/parameter, 256 | ditto, 1024 |
|---|---|---|---|---|---|
| 12 | 2.27 GB | 70.8M | 13.9 | 352 | 88 |
| **32** | **6.05 GB** | **189M** | **5.2** | **940** | **235** |
| 60 | 11.3 GB | 354M | 2.8 | 1763 | 441 |

The 256-wide net has ~201k parameters and would be fine on 12 shards. The 1024-wide variant has
~804k, and at 12 shards it would see 88 positions per parameter against the 256-wide net's 352 --
so a width comparison run on that data would be measuring which net is least starved rather than
which architecture is better. 32 shards is the knee: half of Drive's 15 GB free tier, five clean
epochs, and no width handicapped. Going to 48 or 60 buys little and risks filling the quota.

The code below is generated from the repository by `tools/make_colab_notebook.py`. Do not edit it
here -- edit the repository and regenerate, or the net you train stops matching the net the engine
expects.

## 1. Check the GPU actually attached

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU -- set Runtime > Change runtime type > T4 GPU")
print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")

## 2. Write the training code

Verbatim copies of the repository modules. `numba` and `torch` are preinstalled on Colab; the
`zstandard`/`orjson` install is only needed if you use the optional preprocessing cell.

In [ ]:
import pathlib
pathlib.Path('training').mkdir(exist_ok=True)

In [ ]:
%%writefile training/__init__.py
"""Offline NNUE training. None of this ships -- it imports torch, zstandard and orjson, none of
which exist in the match container, and `harness/package.py` never collects this directory."""

In [ ]:
%%writefile training/dataset.py
"""Stream the packed shards into training batches.

Nothing here ships. The decode kernel is jitted for the same reason the engine is: a batch is
16k positions of roughly 24 pieces each, so a step needs ~400k feature indices, and building those
in a Python loop would cost more than the gradient step it feeds.

The two perspectives are the whole point of the architecture. Every position produces two feature
sets over the same 768 inputs -- one as white sees it, one as black sees it, the latter with
colours swapped and squares vertically mirrored. The network is then fed
`[side-to-move accumulator, other accumulator]`, which makes it structurally colour-symmetric: a
position and its mirror image produce identical activations. That matters here specifically,
because the source data is skewed toward white (mean +210 cp, and the material distribution is
itself lopsided). A net taking white-relative inputs would learn that skew as a bias; this one
cannot represent it, provided the target is also converted to side-to-move relative, which
`load_batch` does.
"""

from __future__ import annotations

from collections.abc import Iterator
from pathlib import Path

import numpy as np
from numba import njit

RECORD_SIZE = 32
FEATURES = 768
BUCKETS = 8

# The centipawn scale of the sigmoid that turns an evaluation into a win probability. 400 is the
# usual figure and is not tuned here; it sets how much the loss cares about the difference between
# +300 and +600, which is very little, against +0 and +100, which is a great deal.
SCALE = 400.0


@njit(
    "int64(uint8[:], int64, int32[:], int32[:], int64[:], uint8[:], uint8[:], int16[:])", cache=True
)
def decode_batch(
    data: np.ndarray,
    count: int,
    white_indices: np.ndarray,
    black_indices: np.ndarray,
    offsets: np.ndarray,
    stm: np.ndarray,
    bucket: np.ndarray,
    score: np.ndarray,
) -> int:
    """Expand packed records into flat feature indices plus per-position offsets.

    Returns the number of feature indices written. Both perspectives always have the same piece
    count, so a single offsets array serves both.
    """
    cursor = 0
    for position in range(count):
        base = position * RECORD_SIZE
        offsets[position] = cursor

        occupancy = np.uint64(0)
        for byte in range(8):
            occupancy |= np.uint64(data[base + byte]) << np.uint64(8 * byte)

        # A straight scan of all 64 squares, not a bit-scan loop: it visits squares in ascending
        # order, which is the order the nibbles were written in, and 64 fixed iterations beat
        # shifting a mask down for each of ~24 pieces.
        slot = 0
        for square in range(64):
            if (occupancy >> np.uint64(square)) & np.uint64(1) == np.uint64(0):
                continue

            packed = data[base + 8 + slot // 2]
            code = np.int64(packed & 0xF) if slot % 2 == 0 else np.int64(packed >> 4)
            slot += 1

            white_indices[cursor] = np.int32(code * 64 + square)
            # Black's view: swap the colour of every piece and mirror the board vertically.
            flipped = code - 6 if code >= 6 else code + 6
            black_indices[cursor] = np.int32(flipped * 64 + (square ^ 56))
            cursor += 1

        score[position] = np.int16(data[base + 24]) | (np.int16(data[base + 25]) << 8)
        stm[position] = data[base + 26]
        bucket[position] = data[base + 27]
    offsets[count] = cursor
    return cursor


class ShardStream:
    """Reads shards in a random order, shuffling within a block before yielding batches.

    Preprocessing already scattered records across shards at random, which is a shuffle across the
    whole file. This adds the second half: a block of records is read, permuted in memory and cut
    into batches. Together they approximate a full shuffle of a file far larger than RAM, without
    ever needing to sort 11 GB.
    """

    def __init__(
        self,
        directory: Path,
        batch_size: int = 16384,
        block_records: int = 1 << 20,
        seed: int = 0,
        holdout: int = 0,
    ) -> None:
        self.shards = sorted(directory.glob("shard*.bin"))
        if not self.shards:
            raise FileNotFoundError(f"no shard*.bin in {directory}")
        # The last few shards are reserved for validation so that no position the net trains on is
        # ever scored against it. Shards are interchangeable, so holding out whole files is enough.
        if holdout:
            self.shards = self.shards[:-holdout]
        self.batch_size = batch_size
        self.block_records = block_records
        self.rng = np.random.default_rng(seed)

    @property
    def total_records(self) -> int:
        return sum(path.stat().st_size // RECORD_SIZE for path in self.shards)

    def blocks(self) -> Iterator[np.ndarray]:
        order = self.rng.permutation(len(self.shards))
        for shard_index in order:
            path = self.shards[shard_index]
            with open(path, "rb") as handle:
                while True:
                    raw = handle.read(self.block_records * RECORD_SIZE)
                    if len(raw) < RECORD_SIZE:
                        break
                    flat = np.frombuffer(raw, dtype=np.uint8)
                    count = len(flat) // RECORD_SIZE
                    block = flat[: count * RECORD_SIZE].reshape(count, RECORD_SIZE)
                    yield block[self.rng.permutation(count)]

    def batches(self) -> Iterator[dict[str, np.ndarray]]:
        for block in self.blocks():
            for start in range(0, len(block) - self.batch_size + 1, self.batch_size):
                yield self._decode(block[start : start + self.batch_size])

    def _decode(self, records: np.ndarray) -> dict[str, np.ndarray]:
        count = len(records)
        flat = np.ascontiguousarray(records).reshape(-1)
        capacity = count * 32
        white = np.empty(capacity, dtype=np.int32)
        black = np.empty(capacity, dtype=np.int32)
        offsets = np.empty(count + 1, dtype=np.int64)
        stm = np.empty(count, dtype=np.uint8)
        bucket = np.empty(count, dtype=np.uint8)
        score = np.empty(count, dtype=np.int16)
        written = int(decode_batch(flat, count, white, black, offsets, stm, bucket, score))
        return {
            "white": white[:written],
            "black": black[:written],
            "offsets": offsets[:count],
            "stm": stm,
            "bucket": bucket,
            # White-relative on disk, side-to-move relative here. See the module docstring: the
            # architecture is colour-symmetric only if the target is too.
            "score": score.astype(np.float32) * np.where(stm == 1, -1.0, 1.0).astype(np.float32),
        }

In [ ]:
%%writefile training/preprocess.py
"""Turn the Lichess evaluations database into fixed-width records the trainer can stream.

Nothing here ships. `harness/package.py` collects root `*.py` plus whatever `--include` names, so
this directory is never in the zip -- which is the point, since it imports `zstandard` and `orjson`
and neither exists in the match container.

Input is `lichess_db_eval.jsonl.zst`: 21.7 GB compressed, 394,669,566 positions, CC0. It is never
expanded to disk (~150 GB) -- the stream is decompressed, parsed and discarded record by record.

Output is 32-byte records written to `SHARDS` files chosen uniformly at random. Random sharding is
half of the shuffle; the trainer shuffles within a shard, and the two together approximate a full
shuffle over a file far larger than memory. The record stores a *position*, not a list of feature
indices: the board costs 24 bytes where 32 indices would cost 64, and expanding it is a few
nanoseconds against a training step measured in milliseconds.

    byte  0.. 7  occupancy bitboard, little-endian uint64
    byte  8..23  one nibble per occupied square in ascending square order, low nibble first;
                 the nibble is the engine's own piece code, 0..11 = WP WN WB WR WQ WK BP .. BK
    byte 24..25  int16 score in centipawns, **white-relative**, clamped to +-2000
    byte 26      side to move, 0 white 1 black
    byte 27      output bucket, 0..7, by piece count
    byte 28..31  reserved, zero

Three properties of the source were established by measurement, not by reading, because all three
are silent if wrong. `notes/measurements.md` records them:

- **`cp` is white-relative, not side-to-move.** Multi-PV lines are ordered best-first for the side
  to move, so white-relative scores run ascending when black is to move. Over 200k sampled
  records: black to move 86,599 ascending against 3 descending, and the mirror image for white.
  Getting this backwards trains a net that plays the opponent's side.
- **`depth >= 20` keeps 91.3%** of positions, so roughly 360M survive.
- **The file contains illegal positions.** Lichess evaluates boards its users set up by hand, so
  record 7,106 alone has seventeen black pieces and three black knights. They are filtered here;
  training on them would spend capacity fitting positions that cannot arise in a game.

Throughput is bounded by zstd, not by us. Decompression of this file runs at ~32 MB/s on one core
against a disk that does 379 MB/s, so the serial floor for a full pass is a bit over an hour. The
parse is therefore pushed to a worker pool and overlapped with it rather than optimised.
"""

from __future__ import annotations

import argparse
import multiprocessing
import random
import struct
import time
from collections.abc import Iterator
from pathlib import Path
from typing import Any

import orjson
import zstandard

# 12 piece types x 64 squares. Not HalfKP: with plain piece-square inputs a king move is an
# ordinary incremental update, where HalfKP would force a full accumulator refresh.
FEATURES = 768

SHARDS = 64
RECORD_SIZE = 32

# Beyond this the position is decided and the exact number carries no information worth fitting.
# Mates map to the clamp rather than being dropped -- "winning" is the signal, not the ply count.
SCORE_CLAMP = 2000

MIN_DEPTH = 20

CHUNK_LINES = 20_000

# Piece letters in the engine's own order, so a feature index computed here and one computed by the
# engine at match time cannot drift apart.
PIECE_CODES = {
    "P": 0, "N": 1, "B": 2, "R": 3, "Q": 4, "K": 5,
    "p": 6, "n": 7, "b": 8, "r": 9, "q": 10, "k": 11,
}  # fmt: skip


def parse_board(field: str) -> tuple[int, list[int]]:
    """Read a FEN placement field into (occupancy, piece codes in ascending square order).

    FEN lists rank 8 first and the engine numbers a1 as square 0, so the ranks are walked in
    reverse. That ordering is not cosmetic: it means the codes come out already sorted by square,
    and a sort of 32 items avoided 394 million times is worth the one-line comment.
    """
    occupancy = 0
    codes: list[int] = []
    square = 0
    for rank in reversed(field.split("/")):
        for character in rank:
            if character.isdigit():
                square += int(character)
            else:
                occupancy |= 1 << square
                codes.append(PIECE_CODES[character])
                square += 1
    return occupancy, codes


def is_plausible(occupancy: int, codes: list[int]) -> bool:
    """Reject positions that could not occur in a game.

    Not a full legality check -- that would need attack generation for a marginal gain. This
    catches the hand-built boards the database actually contains: wrong king count, too many
    pieces or pawns of a colour, and pawns on the back ranks.
    """
    if len(codes) > 32:
        return False
    white = black = white_pawns = black_pawns = white_kings = black_kings = 0
    remaining = occupancy
    for code in codes:
        square = (remaining & -remaining).bit_length() - 1
        remaining &= remaining - 1
        if code < 6:
            white += 1
            if code == 0:
                white_pawns += 1
                if square < 8 or square >= 56:
                    return False
            elif code == 5:
                white_kings += 1
        else:
            black += 1
            if code == 6:
                black_pawns += 1
                if square < 8 or square >= 56:
                    return False
            elif code == 11:
                black_kings += 1
    return (
        white_kings == 1
        and black_kings == 1
        and white <= 16
        and black <= 16
        and white_pawns <= 8
        and black_pawns <= 8
    )


def encode(occupancy: int, codes: list[int], score: int, stm: int) -> bytes:
    """Pack one position into its 32 bytes."""
    nibbles = bytearray(16)
    for index, code in enumerate(codes):
        if index % 2 == 0:
            nibbles[index // 2] |= code
        else:
            nibbles[index // 2] |= code << 4
    # Eight buckets over 2..32 pieces. The net learns a separate output head per bucket, which is
    # most of what a phase-tapered evaluation buys, learned rather than hand-set.
    bucket = min(7, max(0, (len(codes) - 2) // 4))
    return (
        struct.pack("<Q", occupancy) + bytes(nibbles) + struct.pack("<hBBI", score, stm, bucket, 0)
    )


def best_score(evals: list[dict[str, Any]]) -> int | None:
    """The deepest evaluation's principal line, as a clamped white-relative centipawn score."""
    best_depth = -1
    chosen = None
    for entry in evals:
        if entry["depth"] > best_depth and entry["pvs"]:
            best_depth = entry["depth"]
            chosen = entry["pvs"][0]
    if chosen is None or best_depth < MIN_DEPTH:
        return None
    if "cp" in chosen:
        return max(-SCORE_CLAMP, min(SCORE_CLAMP, int(chosen["cp"])))
    mate = int(chosen["mate"])
    # A mate of 0 means the game is already over; treat it as unusable rather than guess a sign.
    if mate == 0:
        return None
    return SCORE_CLAMP if mate > 0 else -SCORE_CLAMP


def process_chunk(job: tuple[int, list[bytes]]) -> tuple[int, list[bytes]]:
    """Parse a batch of raw JSON lines into per-shard blobs. Runs in a worker process.

    The shard is drawn here rather than in the parent so the randomness costs nothing serial. It is
    seeded from the chunk index, so a re-run with the same input reproduces the same split.
    """
    index, lines = job
    rng = random.Random(index)
    buckets: list[bytearray] = [bytearray() for _ in range(SHARDS)]
    kept = 0
    for line in lines:
        record = orjson.loads(line)
        score = best_score(record["evals"])
        if score is None:
            continue
        fen = record["fen"]
        placement, _, rest = fen.partition(" ")
        occupancy, codes = parse_board(placement)
        if not is_plausible(occupancy, codes):
            continue
        buckets[rng.randrange(SHARDS)] += encode(
            occupancy, codes, score, 1 if rest[0] == "b" else 0
        )
        kept += 1
    return kept, [bytes(bucket) for bucket in buckets]


def read_chunks(source: Path, limit: int) -> Iterator[tuple[int, list[bytes]]]:
    """Decompress and split into batches of lines. This is the serial bottleneck; keep it bare."""
    index = 0
    read = 0
    tail = b""
    pending: list[bytes] = []
    with open(source, "rb") as raw:
        reader = zstandard.ZstdDecompressor().stream_reader(raw)
        while True:
            block = reader.read(1 << 22)
            if not block:
                break
            lines = (tail + block).split(b"\n")
            tail = lines.pop()
            pending.extend(lines)
            read += len(lines)
            while len(pending) >= CHUNK_LINES:
                yield index, pending[:CHUNK_LINES]
                pending = pending[CHUNK_LINES:]
                index += 1
            if limit and read >= limit:
                break
    if pending:
        yield index, pending


def run(source: Path, destination: Path, limit: int, workers: int) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    # Sixty-four files held open for the whole pass, which is why they are not context-managed
    # individually; the `finally` below closes them. Nesting 64 `with` blocks would say the same
    # thing at far greater length.
    handles = [
        open(destination / f"shard{shard:02d}.bin", "wb", buffering=1 << 22)  # noqa: SIM115
        for shard in range(SHARDS)
    ]

    written = 0
    chunks = 0
    started = time.perf_counter()
    try:
        with multiprocessing.Pool(workers) as pool:
            for kept, blobs in pool.imap(process_chunk, read_chunks(source, limit), chunksize=1):
                written += kept
                chunks += 1
                for handle, blob in zip(handles, blobs, strict=True):
                    if blob:
                        handle.write(blob)
                if chunks % 250 == 0:
                    elapsed = time.perf_counter() - started
                    read = chunks * CHUNK_LINES
                    print(
                        f"  {read:,} read  {written:,} kept  {read / elapsed:,.0f}/s"
                        f"  {written * RECORD_SIZE / 1e9:.2f} GB  {elapsed / 60:.0f}m",
                        flush=True,
                    )
    finally:
        for handle in handles:
            handle.close()

    elapsed = time.perf_counter() - started
    read = chunks * CHUNK_LINES
    print(f"read ~{read:,}, kept {written:,} in {elapsed / 60:.1f} minutes")
    print(f"{written * RECORD_SIZE / 1e9:.2f} GB across {SHARDS} shards in {destination}")


def main() -> None:
    parser = argparse.ArgumentParser(description="Preprocess the Lichess eval database.")
    parser.add_argument(
        "--source", type=Path, default=Path(r"C:/Users/ssjag/chessdata/lichess_db_eval.jsonl.zst")
    )
    parser.add_argument(
        "--destination", type=Path, default=Path(r"C:/Users/ssjag/chessdata/shards")
    )
    parser.add_argument("--limit", type=int, default=0, help="stop after this many input records")
    parser.add_argument(
        "--workers", type=int, default=max(1, (multiprocessing.cpu_count() * 3) // 4)
    )
    arguments = parser.parse_args()
    run(arguments.source, arguments.destination, arguments.limit, arguments.workers)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile training/train.py
"""Train the NNUE and export int16 weights.

Nothing here ships -- torch never gets imported in the match container. The output is a `.npz` of
quantised integer arrays, which `engine/` loads and evaluates with numba.

Architecture: `(768 -> HIDDEN) x 2 -> 1`, SCReLU, eight output buckets by piece count.

Deliberately *not* HalfKP. Three of HalfKP's four problems are properties of the shipped net rather
than of training, so no amount of cloud GPU fixes them: a 21 MB table has poor locality on one
core shared with an opponent hammering the same L3, king moves force a full accumulator refresh
over that table, and numba cannot emit the hand-tuned int8 AVX2 that makes HalfKP pay in C++.
With plain 768 inputs there is no king refresh at all -- every move is an incremental update.

Quantisation, and why the scales are what they are
--------------------------------------------------
`QA = 255` scales the feature transformer, `QB = 64` the output layer. SCReLU squares a value
clamped to `[0, QA]`, so the square reaches `QA^2 = 65025` and must live in int32; the accumulator
itself is int16, which is what makes the incremental update cheap.

    accumulator     int16, QA * float
    screlu          clamp(acc, 0, QA)^2       -> int32, QA^2 * float
    output weights  int16, QB * float
    eval_cp         (sum(screlu * w) / QA + bias * QA * QB) * SCALE / (QA * QB)

Weights are clamped every step rather than only at export, so the network trains inside the range
it will be quantised into instead of being lopped off at the end. The output weights are held to
`+-127/QB`, which is the widest value `int16 * QB` can represent without risking overflow once
summed over the hidden layer.

None of the above is trusted. `validate_quantisation` scores held-out positions in float and in
integer arithmetic and reports the worst divergence; a few centipawns is quantisation noise, tens
of centipawns is a scale bug. That check is the gate, because a wrong shift here is otherwise
completely silent -- the net simply plays slightly worse, forever.
"""

from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np
import torch
from torch import Tensor, nn

from training.dataset import BUCKETS, FEATURES, SCALE, ShardStream

QA = 255
QB = 64

# The widest output weight that survives quantisation to int16 at QB.
OUTPUT_CLAMP = 127.0 / QB


def screlu(x: Tensor) -> Tensor:
    """Squared clipped ReLU. The square is what gives the net a cheap non-linearity that still
    quantises exactly, since clamping to [0, 1] before squaring keeps everything in range."""
    return torch.clamp(x, 0.0, 1.0) ** 2


class Nnue(nn.Module):
    def __init__(self, hidden: int = 256) -> None:
        super().__init__()
        self.hidden = hidden
        # sparse sum over the active features -- exactly what the incremental accumulator does at
        # match time, so training and inference compute the same quantity by construction.
        self.transformer = nn.EmbeddingBag(FEATURES, hidden, mode="sum")
        self.transformer_bias = nn.Parameter(torch.zeros(hidden))
        self.output = nn.Parameter(torch.zeros(BUCKETS, 2 * hidden))
        self.output_bias = nn.Parameter(torch.zeros(BUCKETS))

        # Random initialisation, and nothing else. Starting from a published chess network is
        # disqualifying, so this is the only initialisation the project is permitted to use.
        bound = 1.0 / np.sqrt(hidden)
        nn.init.uniform_(self.transformer.weight, -bound, bound)
        nn.init.uniform_(self.output, -bound, bound)

    def forward(
        self,
        white: Tensor,
        black: Tensor,
        offsets: Tensor,
        stm: Tensor,
        bucket: Tensor,
    ) -> Tensor:
        white_acc = self.transformer(white, offsets) + self.transformer_bias
        black_acc = self.transformer(black, offsets) + self.transformer_bias

        side = stm.unsqueeze(1).to(white_acc.dtype)
        # stm == 0 means white to move, so `us` is the white accumulator then.
        us = white_acc * (1.0 - side) + black_acc * side
        them = black_acc * (1.0 - side) + white_acc * side

        hidden = screlu(torch.cat([us, them], dim=1))
        weights = self.output[bucket]
        return (hidden * weights).sum(dim=1) + self.output_bias[bucket]

    @torch.no_grad()
    def clamp_weights(self) -> None:
        """Hold the parameters inside the range quantisation can represent."""
        self.output.clamp_(-OUTPUT_CLAMP, OUTPUT_CLAMP)
        # The accumulator is int16 and holds QA * (sum of ~32 weights plus the bias). Bounding each
        # weight well below 127/QA leaves room for that sum without ever overflowing.
        self.transformer.weight.clamp_(-1.98, 1.98)
        self.transformer_bias.clamp_(-1.98, 1.98)


def to_tensors(batch: dict[str, np.ndarray], device: torch.device) -> dict[str, Tensor]:
    return {
        "white": torch.from_numpy(batch["white"].astype(np.int64)).to(device),
        "black": torch.from_numpy(batch["black"].astype(np.int64)).to(device),
        "offsets": torch.from_numpy(batch["offsets"]).to(device),
        "stm": torch.from_numpy(batch["stm"].astype(np.int64)).to(device),
        "bucket": torch.from_numpy(batch["bucket"].astype(np.int64)).to(device),
        "score": torch.from_numpy(batch["score"]).to(device),
    }


def loss_of(prediction: Tensor, score: Tensor) -> Tensor:
    """Mean squared error in win-probability space, not in centipawns.

    Centipawn error weights a blunder from +900 to +1200 as heavily as one from +0 to +300, when
    only the second changes the result. Passing both through a sigmoid at `SCALE` fixes that, and
    is why the loss is computed here rather than with a bare MSE on the raw evaluation.
    """
    return ((torch.sigmoid(prediction) - torch.sigmoid(score / SCALE)) ** 2).mean()


def quantise(model: Nnue) -> dict[str, np.ndarray]:
    """Integer weights in the layout the engine reads."""
    with torch.no_grad():
        transformer = model.transformer.weight.detach().cpu().numpy()
        return {
            # Left as [features, hidden], which is what torch already stores, so that the row for
            # one feature is 256 contiguous int16. That is the only layout the accumulator wants:
            # a move changes a handful of features and each one is then a single 512-byte run the
            # engine adds or subtracts. Transposing to [hidden, features] would put a feature on a
            # 1536-byte stride and cost a cache line per element.
            "transformer": np.round(transformer * QA).astype(np.int16),
            "transformer_bias": np.round(model.transformer_bias.detach().cpu().numpy() * QA).astype(
                np.int16
            ),
            "output": np.round(model.output.detach().cpu().numpy() * QB).astype(np.int16),
            "output_bias": np.round(model.output_bias.detach().cpu().numpy() * QA * QB).astype(
                np.int32
            ),
            "meta": np.array([model.hidden, QA, QB, int(SCALE)], dtype=np.int32),
        }


def save_weights(path: Path, weights: dict[str, np.ndarray]) -> None:
    """Write the quantised arrays. numpy's `savez` stub types its keyword arguments as `bool`,
    which is what the ignore is for -- the call itself is the documented usage."""
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(path, **weights)  # type: ignore[arg-type]


def integer_eval(weights: dict[str, np.ndarray], batch: dict[str, np.ndarray]) -> np.ndarray:
    """Score a batch using only integer arithmetic, exactly as the engine will.

    This exists to be compared against the float model. It is the only check that the quantisation
    scales are right, and it is deliberately written from the documented formula rather than by
    reusing anything from `quantise`.
    """
    transformer = weights["transformer"].astype(np.int32)  # [features, hidden]
    transformer_bias = weights["transformer_bias"].astype(np.int32)
    output = weights["output"].astype(np.int32)
    output_bias = weights["output_bias"].astype(np.int64)
    hidden = int(weights["meta"][0])

    count = len(batch["stm"])
    offsets = np.append(batch["offsets"], len(batch["white"]))
    results = np.empty(count, dtype=np.float64)

    for position in range(count):
        start, end = int(offsets[position]), int(offsets[position + 1])
        white_acc = transformer_bias + transformer[batch["white"][start:end]].sum(axis=0)
        black_acc = transformer_bias + transformer[batch["black"][start:end]].sum(axis=0)
        if int(batch["stm"][position]) == 0:
            accumulator = np.concatenate([white_acc, black_acc])
        else:
            accumulator = np.concatenate([black_acc, white_acc])

        clipped = np.clip(accumulator, 0, QA).astype(np.int64)
        activated = clipped * clipped
        total = int((activated * output[int(batch["bucket"][position])].astype(np.int64)).sum())
        total = total // QA + int(output_bias[int(batch["bucket"][position])])
        results[position] = total * SCALE / (QA * QB)

    assert hidden * 2 == output.shape[1]
    return results


def validate_quantisation(
    model: Nnue, batch: dict[str, np.ndarray], device: torch.device
) -> tuple[float, float]:
    """Worst and mean divergence, in centipawns, between the float net and the integer net."""
    model.eval()
    tensors = to_tensors(batch, device)
    with torch.no_grad():
        reference = model(
            tensors["white"],
            tensors["black"],
            tensors["offsets"],
            tensors["stm"],
            tensors["bucket"],
        )
        reference_cp = (reference * SCALE).cpu().numpy().astype(np.float64)
    model.train()
    integer_cp = integer_eval(quantise(model), batch)
    difference = np.abs(reference_cp - integer_cp)
    return float(difference.max()), float(difference.mean())


def train(
    shards: Path,
    output: Path,
    hidden: int,
    batch_size: int,
    steps: int,
    learning_rate: float,
    device_name: str,
    holdout: int,
) -> None:
    device = torch.device(device_name)
    model = Nnue(hidden).to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=max(1, steps))

    stream = ShardStream(shards, batch_size=batch_size, seed=1, holdout=holdout)
    validation = next(ShardStream(shards, batch_size=8192, seed=99, holdout=0).batches())

    # flush=True on every print in this file, not just the periodic ones. A run this long is
    # watched through a redirected log, and Python block-buffers a pipe: without it the opening
    # lines sit invisible for the first ten minutes and a run that died at startup is
    # indistinguishable from one that is working.
    print(
        f"{stream.total_records:,} training records across {len(stream.shards)} shards", flush=True
    )
    print(f"hidden {hidden}, batch {batch_size}, {steps:,} steps on {device_name}", flush=True)

    output.parent.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    running = 0.0
    step = 0
    for step, batch in enumerate(stream.batches(), start=1):
        tensors = to_tensors(batch, device)
        prediction = model(
            tensors["white"],
            tensors["black"],
            tensors["offsets"],
            tensors["stm"],
            tensors["bucket"],
        )
        loss = loss_of(prediction, tensors["score"])

        optimiser.zero_grad(set_to_none=True)
        loss.backward()  # type: ignore[no-untyped-call]
        optimiser.step()
        schedule.step()
        model.clamp_weights()

        running += float(loss.detach())
        if step % 200 == 0:
            elapsed = time.perf_counter() - started
            rate = step * batch_size / elapsed
            print(
                f"  step {step:>7,}/{steps:,}  loss {running / 200:.5f}"
                f"  {rate:,.0f} pos/s  {elapsed / 60:.1f}m",
                flush=True,
            )
            running = 0.0
        if step % 5000 == 0 or step == steps:
            # Checkpoint often. A free Colab session can be killed at any moment, and an epoch of
            # this size is hours -- losing one is losing an evening.
            worst, mean = validate_quantisation(model, validation, device)
            print(
                f"  quantisation divergence: worst {worst:.1f} cp, mean {mean:.2f} cp", flush=True
            )
            save_weights(output, quantise(model))
            torch.save(model.state_dict(), output.with_suffix(".pt"))
            print(f"  saved {output}", flush=True)
        if step >= steps:
            break

    worst, mean = validate_quantisation(model, validation, device)
    print(f"final quantisation divergence: worst {worst:.1f} cp, mean {mean:.2f} cp", flush=True)
    save_weights(output, quantise(model))
    torch.save(model.state_dict(), output.with_suffix(".pt"))
    print(f"wrote {output} ({output.stat().st_size / 1024:.0f} KB)", flush=True)


def main() -> None:
    parser = argparse.ArgumentParser(description="Train the NNUE.")
    parser.add_argument("--shards", type=Path, default=Path(r"C:/Users/ssjag/chessdata/shards"))
    parser.add_argument(
        "--output", type=Path, default=Path(r"C:/Users/ssjag/chessdata/nets/net.npz")
    )
    parser.add_argument("--hidden", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=16384)
    parser.add_argument("--steps", type=int, default=60_000)
    parser.add_argument("--learning-rate", type=float, default=1e-3)
    parser.add_argument(
        "--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu"
    )
    parser.add_argument("--holdout", type=int, default=4, help="shards reserved for validation")
    arguments = parser.parse_args()
    train(
        arguments.shards,
        arguments.output,
        arguments.hidden,
        arguments.batch_size,
        arguments.steps,
        arguments.learning_rate,
        arguments.device,
        arguments.holdout,
    )


if __name__ == "__main__":
    main()

## 3. Mount Drive and confirm the data is there

In [ ]:
from google.colab import drive
from pathlib import Path

from training.dataset import RECORD_SIZE

drive.mount('/content/drive')

shards = Path('/content/drive/MyDrive/chessathon/shards')
nets = Path('/content/drive/MyDrive/chessathon/nets')
nets.mkdir(parents=True, exist_ok=True)

files = sorted(shards.glob('shard*.bin'))
assert files, f"no shard*.bin in {shards} -- upload some first"

total = sum(path.stat().st_size for path in files)
positions = total // RECORD_SIZE
print(f"{len(files)} shards, {total / 1e9:.2f} GB, {positions:,} positions")
print(f"60,000 steps x 16,384 = {60000 * 16384 / positions:.1f} epochs over this data")

## 4. Throughput probe -- run this before committing to a long run

400 steps, to check the GPU is doing what the profile says it should. Measured on the laptop, a
step at batch 16,384 is **98.5% forward/backward** and only 24.7 ms of CPU work -- read, decode and
tensor construction. So the CPU floor is about **664,000 pos/s** and everything below that is the
GPU's to win. Two vCPUs are not a constraint: the decode is a single serial thread of about 18 ms,
so core *count* is irrelevant here.

Expect comfortably over 100,000 pos/s. Under about 50,000 means something is wrong -- no GPU
actually attached, or Drive I/O throttling the shard reads -- and is worth fixing before spending
an hour on the real run rather than after.

In [ ]:
!python -m training.train \
    --shards /content/drive/MyDrive/chessathon/shards \
    --output /content/probe.npz \
    --steps 400 --holdout 2

## 5. The real run

Checkpoints every 5,000 steps straight to Drive, so a reclaimed session costs the steps since the
last checkpoint rather than the whole run.

Leave `--steps 60000` alone whatever you uploaded. It is the total number of *samples* the
optimiser sees that matters, and 60,000 x 16,384 = 983M is the schedule the cosine learning-rate
decay is built around -- cutting it short leaves the run stranded at a high learning rate. On 32
shards that is 5.2 epochs, which is a normal number of passes for a net this small.

In [ ]:
!python -m training.train \
    --shards /content/drive/MyDrive/chessathon/shards \
    --output /content/drive/MyDrive/chessathon/nets/net256.npz \
    --hidden 256 --steps 60000 --holdout 2

## 6. Width A/B

The one number the plan says we must measure ourselves, because published width deltas come from
C++ engines at different time controls and do not transfer. All four write separate files; SPRT
picks the winner in the engine, at our own time control. This is the actual reason to be on a GPU
at all -- it is four runs, not one.

In [ ]:
# One line per run on purpose: IPython's `!` takes a single line, and backslash continuation
# inside a loop body is not reliably joined before the shell sees it.
for width in (128, 512, 1024):
    print(f"=== hidden {width} ===", flush=True)
    !python -m training.train --shards /content/drive/MyDrive/chessathon/shards --output /content/drive/MyDrive/chessathon/nets/net{width}.npz --hidden {width} --steps 60000 --holdout 2

## 7. Bring the weights home

`net256.npz` is about 400 KB. It goes to `weights/nnue.npz` in the repository, where
`harness/package.py` already picks up a root-level `weights` directory by default.

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/chessathon/nets/net256.npz')

## Optional: rebuild the shards here instead of uploading them

Only worth it if uploading is impossible. Colab downloads the 21.7 GB database at datacenter speed,
but preprocessing is CPU-bound and a free instance has two vCPUs against the laptop's nine workers,
so this is slower than uploading a subset over most home connections. `--limit` takes a *prefix* of
the database rather than a random sample, which is a real caveat: the ordering is not documented,
so a prefix is not guaranteed to be representative the way a subset of shards is.

In [ ]:
!pip install -q zstandard orjson
!wget -c https://database.lichess.org/lichess_db_eval.jsonl.zst -O /content/eval.jsonl.zst
!python -m training.preprocess \
    --input /content/eval.jsonl.zst \
    --output /content/drive/MyDrive/chessathon/shards \
    --limit 100000000